# 02 · Transaction categorizer

In [1]:
import json
import pandas as pd
from src.utils import MODELS_DIR, CATEGORIZED_CSV
from src.train_categorizer import predict_category

metrics = json.load(open(MODELS_DIR / 'categorizer_metrics.json', encoding='utf-8'))
pd.DataFrame({k: v for k, v in metrics.items() if isinstance(v, dict) and 'macro_f1' in v}).T

,accuracy,macro_f1
baseline_majority,0.4502,0.0690
baseline_keyword_rules,0.7942,0.7506
logreg,0.9550,0.9488
linear_svc,0.9518,0.9459
complement_nb,0.9518,0.9465
logreg_tuned,0.9518,0.9419
hidden_truth_on_unlabelled,0.9469,0.9289


In [2]:
out = pd.read_csv(CATEGORIZED_CSV, parse_dates=['date'])
review = out[out['needs_review']].sort_values('confidence')
review[['date', 'description', 'amount', 'predicted_category', 'confidence']].head(20)

,date,description,amount,predicted_category,confidence
408,2024-04-15,POS LIAM KOSHY HYDERABAD,411.40,Transfers,0.2220
2095,2026-08-17,POS GPAY Delhi,586.96,Transfers,0.2282
495,2024-06-02,NEFT-456649-BHIM UPI,551.97,Shopping,0.2391
170,2023-12-24,IMPS/39536/BHIM UPI,427.83,Shopping,0.2391
1842,2026-04-12,UPI-BHIM UPI-970727@okaxis,306.14,Shopping,0.2391
198,2024-01-11,UPI-BHIM UPI-683613@okaxis,511.13,Shopping,0.2391
181,2024-01-01,UPI/GPAY/368700/PAY,574.27,Transfers,0.2543
1702,2026-02-06,UPI/GPAY/484238/PAY,780.09,Transfers,0.2543
1947,2026-06-07,GPAY*309264,622.43,Transfers,0.2543
1054,2025-03-24,Quincy Biswas ORDER 885691,124.47,Food,0.2639


In [3]:
for text, amt in [('UPI/SWIGGY/12345/PAY', 350), ('UBER INDIA*9981', 220), ('Apollo Pharmacy POS Pune', 640)]:
    print(text, '->', predict_category(text, amt))

UPI/SWIGGY/12345/PAY -> ('Food', 0.8854848916077223)
UBER INDIA*9981 -> ('Transport', 0.9037300433082905)
Apollo Pharmacy POS Pune -> ('Health', 0.9403013313793389)


## My analysis
- Which classes are confused most often, and why?  
  The model struggles most with transactions containing generic wallet or payment gateway descriptors like "PAYTM", "PHONEPE", "GPAY", or plain UPI transfers. Because these descriptions lack specific merchant keywords (like "Zomato" or "Netflix"), the text model doesn't have enough contextual clues to confidently assign a category, leading to confusion between general spending categories.
- What happens if you remove the amount feature?  
  Removing the amount feature would likely decrease the model's accuracy for merchants that span multiple categories. For example, a ₹199 charge from "Amazon" is likely an "Entertainment" expense (Prime subscription), whereas a ₹15,000 charge from "Amazon" is likely "Shopping". The amount feature provides a crucial mathematical signal when the text description is ambiguous.
- What threshold would you pick for manual review?  
  A threshold around 0.60 to 0.70 seems optimal for personal finance. Setting it too high (e.g., 0.95) would require manual review for too many transactions, defeating the purpose of automation. Setting it too low (e.g., 0.40) risks confidently miscategorizing transactions, which could ruin the accuracy of a monthly budget. The 0.60 mark provides a solid balance, automating the obvious items while flagging the genuinely ambiguous ones for human review.